# 6.2 Sentiment Classification

這份 Notebook 使用 `TextVectorization + Embedding` 建立二元情緒分類模型。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

tf.keras.utils.set_random_seed(42)


## 2. 建立正負向評論資料


In [ ]:
def make_sentiment_data(repeats=90, seed=42):
    rng = np.random.default_rng(seed)
    positive_templates = [
        'the product is great and useful',
        'the service is fast and friendly',
        'the tutorial is clear and helpful',
        'the model works well today',
        'the result is excellent and reliable',
        'the app feels smooth and stable',
    ]
    negative_templates = [
        'the product is bad and broken',
        'the service is slow and rude',
        'the tutorial is confusing and unclear',
        'the model fails again today',
        'the result is wrong and unstable',
        'the app feels buggy and slow',
    ]
    suffixes = ['now', 'for me', 'in this test', 'with this update', 'on my device']
    texts, labels = [], []
    for _ in range(repeats):
        for sentence in positive_templates:
            texts.append(sentence + ' ' + rng.choice(suffixes))
            labels.append(1)
        for sentence in negative_templates:
            texts.append(sentence + ' ' + rng.choice(suffixes))
            labels.append(0)
    texts = np.array(texts, dtype=object)
    labels = np.array(labels, dtype='int64')
    idx = rng.permutation(len(labels))
    return texts[idx], labels[idx]

texts, labels = make_sentiment_data(repeats=90, seed=2)
x_train, x_temp, y_train, y_temp = train_test_split(texts, labels, test_size=0.3, random_state=42, stratify=labels)
x_valid, x_test, y_valid, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(len(x_train), len(x_valid), len(x_test))
pd.Series(y_train).value_counts().rename(index={0: 'negative', 1: 'positive'})


## 3. 建立 TextVectorization


In [ ]:
MAX_TOKENS = 1200
SEQ_LEN = 12
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=SEQ_LEN,
)
vectorizer.adapt(x_train)
print(vectorizer.get_vocabulary()[:20])


## 4. 建立情緒分類模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(MAX_TOKENS, 24),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


## 5. 訓練模型


In [ ]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_valid, y_valid),
    epochs=15,
    batch_size=32,
    verbose=0,
)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Sentiment accuracy')
plt.ylim(0, 1.05)
plt.show()


## 6. 評估與預測


In [ ]:
y_prob = model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype('int64')
print(model.evaluate(x_test, y_test, verbose=0, return_dict=True))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))


## 7. 預測新句子


In [ ]:
new_texts = np.array([
    'the product is useful and reliable',
    'the service is slow and confusing',
    'the app feels smooth today',
], dtype=object)
pd.DataFrame({'text': new_texts, 'positive_probability': model.predict(new_texts, verbose=0).ravel()})


## 8. 如何套用自己的資料

把文字欄位整理成 `texts`，標籤整理成 `0/1`。若句子更長，調整 `SEQ_LEN`；若詞彙更多，調整 `MAX_TOKENS`。


## 9. 小結

情緒分類流程可以作為許多文字二元分類任務的模板：文字向量化、Embedding、句子向量、二元輸出與分類評估。
